# Extract satellite patches from the Mesogeos zarr cube (date-grouped, resumable)

Cuts a `WIN x WIN` window (NDVI, LAI, LST day, LST night) around each Track A
sample's cell on its last observed day, for the ViT branch (Models A, C, D).

**Fast path:** samples are grouped by date so each day's map is read once, not
once per sample. Resumable: each date-batch is a shard; re-running skips finished ones.

**Before running (one-time):** open the shared
[mesogeos Drive folder](https://drive.google.com/drive/folders/1aRXQXVvw6hz0eYgtJDoixjPQO-_bRKz9),
right-click `mesogeos_cube.zarr` -> Organise -> Add shortcut -> My Drive. Then Run all.

In [ ]:
!pip -q install zarr xarray
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, pandas as pd, xarray as xr, time
from pathlib import Path

WIN, VARS = 64, ['ndvi', 'lai', 'lst_day', 'lst_night']
H = WIN // 2
CUBE = '/content/drive/MyDrive/mesogeos_cube.zarr'
OUT = Path('/content/drive/MyDrive/wildfire_patches'); OUT.mkdir(exist_ok=True)
MANIFEST = 'https://raw.githubusercontent.com/Aaffnnaann/wildfire-dissertation/main/manifest.csv'

ds = xr.open_zarr(CUBE, consolidated=True)
xs, ys = ds.x.values, ds.y.values
mf = pd.read_csv(MANIFEST, parse_dates=['date'])
mf['ix'] = np.abs(xs[None, :] - mf['lon'].values[:, None]).argmin(1)
mf['iy'] = np.abs(ys[None, :] - mf['lat'].values[:, None]).argmin(1)
dates = np.sort(mf['date'].unique())
print(len(dates), 'unique dates for', len(mf), 'samples',
      f'(avg {len(mf)/len(dates):.1f} samples/date)')

In [ ]:
# timing probe over 3 DATES (not samples) — this is the number that matters now
t0 = time.time()
for d in dates[:3]:
    _ = ds[VARS].sel(time=d, method='nearest').load()
per = (time.time() - t0) / 3
print(f'{per:.1f}s/date  ->  full run ~{per * len(dates) / 3600:.1f}h over {len(dates)} dates')

In [ ]:
# PARALLEL extraction — Drive reads are latency-bound, so overlap them.
# Chunking is (1, full-spatial): 1 date = 1 chunk, so date-grouping is optimal;
# concurrency across dates is where the speedup comes from. Resumable per shard.
from concurrent.futures import ThreadPoolExecutor

WORKERS, BATCH = 8, 400

def cut(arr, iy, ix):
    Y, X = arr.shape[:2]
    out = np.full((WIN, WIN, arr.shape[2]), np.nan, np.float32)
    y0, y1, x0, x1 = iy - H, iy + H, ix - H, ix + H
    cy0, cy1 = max(0, y0), min(Y, y1)
    cx0, cx1 = max(0, x0), min(X, x1)
    out[cy0 - y0:cy1 - y0, cx0 - x0:cx1 - x0] = arr[cy0:cy1, cx0:cx1]
    return out

def load_date(d):
    for t in range(3):
        try:
            day = ds[VARS].sel(time=d, method='nearest')
            return d, np.stack([day[v].values for v in VARS], -1).astype(np.float32)
        except Exception:
            if t == 2: raise
            time.sleep(3)

t0 = time.time()
for b0 in range(0, len(dates), BATCH):
    shard = OUT / f'shard_{b0:06d}.npz'
    if shard.exists():
        continue
    bd = dates[b0:b0 + BATCH]
    rows = mf[mf['date'].isin(bd)].reset_index(drop=True)
    patches = np.full((len(rows), WIN, WIN, len(VARS)), np.nan, np.float32)
    for g0 in range(0, len(bd), WORKERS):
        grp = bd[g0:g0 + WORKERS]
        with ThreadPoolExecutor(len(grp)) as ex:
            for d, arr in ex.map(load_date, grp):
                for j in np.where((rows['date'] == d).values)[0]:
                    patches[j] = cut(arr, int(rows.iy[j]), int(rows.ix[j]))
    np.savez_compressed(shard, patches=patches, idx=rows['idx'].values,
                        split=rows['split'].values.astype('U8'))
    done = min(b0 + BATCH, len(dates))
    el = time.time() - t0
    print(f'{done}/{len(dates)} dates  {el:.0f}s  '
          f'eta {el / done * (len(dates) - done) / 3600:.1f}h  -> {shard.name}')
print('extraction done')

In [ ]:
# assemble date-batch shards into per-split npz (the format the training code expects)
buckets = {}
for s in sorted(OUT.glob('shard_*.npz')):
    d = np.load(s, allow_pickle=True)
    for sp in np.unique(d['split']):
        m = d['split'] == sp
        buckets.setdefault(str(sp), []).append((d['idx'][m], d['patches'][m]))

for sp, parts in buckets.items():
    n = int((mf['split'] == sp).sum())
    full = np.full((n, WIN, WIN, len(VARS)), np.nan, np.float32)
    for idx, pat in parts:
        full[idx] = pat
    np.savez_compressed(OUT / f'{sp}.npz', patches=full, idx=np.arange(n))
    print(sp, full.shape, 'nan%', round(float(np.isnan(full).mean()), 3))

In [ ]:
# sanity check: one sample's 4 channels
import matplotlib.pyplot as plt
d = np.load(OUT / 'train.npz')
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, k, v in zip(axes, range(4), VARS):
    ax.imshow(d['patches'][0, :, :, k]); ax.set_title(v); ax.axis('off')
plt.show()